<a href="https://colab.research.google.com/github/weagan/SSM-and-Mamba/blob/main/True_Mamba_Architecture_for_Shakespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


##True Mamba Architecture for Shakespeare Text Generation
Implements selective state spaces with input-dependent parameters
Supports CPU, single GPU, or multi-GPU training



#how could I see that context is being encoded?
You can observe context encoding at two primary stages in the Mamba architecture:

Local Context through 1D Convolution: The self.conv1d layer processes the input x to capture short-range dependencies. By inspecting the tensor before and after this operation, you can see its transformation.
Long-range Context through Selective Scan: The selective_scan_sequential function maintains a hidden state h that accumulates contextual information over time. This state is selectively updated based on the input-dependent parameters delta, B, and C. The output y of the selective scan then reflects this encoded context.
To 'see' this in action, I'll modify the MambaBlock and selective_scan_sequential code to include print statements that display the shape and some statistics (mean, standard deviation) of the relevant tensors at these critical points. This will allow you to track how the information is being processed and transformed.

After running the modified code, look for print statements starting with [DEBUG CONV] and [DEBUG SCAN] in the output.

#where is Local Context through 1D Convolution coded into h?
The local context from the 1D convolution is encoded into h within the selective_scan_sequential function. Here's a breakdown:

In MambaBlock: The self.conv1d layer processes the input x to capture local context. The output of this convolution is still named x.
Passing to selective_scan_sequential: This x (which now holds the local context) is then passed as the u argument to the selective_scan_sequential function.
In selective_scan_sequential: Inside the loop, at each time step t, u_t (which is u[:, t, :]) represents the current input, carrying the local contextual information. The hidden state h is updated using this u_t in the line: h = deltaA * h + deltaB * u_t.unsqueeze(-1).
Therefore, the local context captured by the 1D convolution is continuously integrated into the hidden state h as the selective scan processes the sequence.



In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import requests
from typing import Optional
import math
import time #really

# =============================================================================
# CONFIGURATION
# =============================================================================

class Config:
    # Model hyperparameters
    d_model = 256          # Hidden dimension size
    d_state = 16           # SSM state dimension (N in paper)
    d_conv = 4             # Convolution kernel size
    expand = 2             # Expansion factor for inner dimension
    n_layers = 4           # Number of Mamba layers
    vocab_size = None      # Will be set based on data

    # Training hyperparameters
    batch_size = 32        # Batch size for training
    seq_length = 512       # Sequence length for training
    learning_rate = 1e-3   # Learning rate (increased for stability)
    num_epochs = 5         # Number of training epochs

    # Dataset configuration
    dataset_fraction = 0.01 # Fraction of dataset to use

    # Device configuration
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    use_multi_gpu = torch.cuda.device_count() > 1

config = Config()

# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

class ShakespeareDataset(Dataset):
    """Dataset for Shakespeare text with character-level tokenization"""

    def __init__(self, text: str, seq_length: int, fraction: float = 1.0):
        """
        Args:
            text: Raw text data
            seq_length: Length of each training sequence
            fraction: Fraction of data to use
        """
        self.seq_length = seq_length

        # Create character-level vocabulary
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)

        # Create mappings between characters and indices
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}

        # Encode entire text as integers
        self.data = [self.char_to_idx[ch] for ch in text]

        # Apply fraction
        self.data = self.data[:int(len(self.data) * fraction)]
        print(f"Using {len(self.data)} characters ({fraction*100:.1f}% of total)")

    def __len__(self):
        """Number of sequences in dataset"""
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        """
        Get a single training example
        Returns:
            x: Input sequence [seq_length]
            y: Target sequence [seq_length] (shifted by 1)
        """
        x = torch.tensor(self.data[idx:idx + self.seq_length], dtype=torch.long)
        y = torch.tensor(self.data[idx + 1:idx + self.seq_length + 1], dtype=torch.long)
        return x, y

def load_shakespeare_data():
    """Download and load Shakespeare dataset"""
    print("Downloading Shakespeare dataset...")
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    response = requests.get(url)
    text = response.text
    print(f"Loaded {len(text)} characters")
    return text

# =============================================================================
# SELECTIVE SCAN - THE CORE OF MAMBA (STABILIZED VERSION)
# =============================================================================

def selective_scan_sequential(u, delta, A, B, C, D):
    """
    Sequential implementation of selective scan (STABILIZED)

    Key fixes for NaN issues:
    1. Clamp delta to prevent exploding exponentials
    2. Use more stable discretization
    3. Add numerical stability checks

    Args:
        u: Input sequence [batch, seq_len, d_inner]
        delta: Time step (input-dependent) [batch, seq_len, d_inner]
        A: State transition [d_inner, d_state]
        B: Input-to-state [batch, seq_len, d_state]
        C: State-to-output [batch, seq_len, d_state]
        D: Skip connection [d_inner]

    Returns:
        y: Output [batch, seq_len, d_inner]
    """
    batch, seq_len, d_inner = u.shape
    d_state = A.shape[1]

    # Initialize hidden state
    h = torch.zeros(batch, d_inner, d_state, device=u.device, dtype=u.dtype)

    outputs = []

    for t in range(seq_len):
        # DEBUG: Inspect hidden state at first time step
        '''
        if t == 0:
            print(f"[DEBUG SCAN] Initial h shape: {h.shape}, mean: {h.mean():.4f}, std: {h.std():.4f}")
        '''
        # Get current inputs
        u_t = u[:, t, :]                    # [batch, d_inner]
        delta_t = delta[:, t, :]             # [batch, d_inner]
        B_t = B[:, t, :].unsqueeze(1)       # [batch, 1, d_state]
        C_t = C[:, t, :].unsqueeze(2)       # [batch, d_state, 1]

        # STABILITY FIX: Clamp delta to prevent numerical issues
        delta_t = torch.clamp(delta_t, min=1e-3, max=0.1)

        # Discretize with stability (use first-order approximation when needed)
        # deltaA = exp(delta * A) \u2248 1 + delta * A for small delta
        dA = delta_t.unsqueeze(-1) * A  # [batch, d_inner, d_state]

        # Use safe exponential with clamping
        deltaA = torch.exp(torch.clamp(dA, min=-10, max=10))

        # deltaB = delta * B
        deltaB = delta_t.unsqueeze(-1) * B_t  # [batch, d_inner, d_state]

        # STATE UPDATE with stability checks
        h = deltaA * h + deltaB * u_t.unsqueeze(-1)

        # Clip hidden state to prevent explosion
        h = torch.clamp(h, min=-1e3, max=1e3)

        # OUTPUT
        y_t = torch.matmul(h, C_t).squeeze(-1) + D * u_t  # [batch, d_inner]

        outputs.append(y_t)

    # Stack outputs
    y = torch.stack(outputs, dim=1)  # [batch, seq_len, d_inner]
    # DEBUG: Inspect final h properties (last time step)
    # print(f"[DEBUG SCAN] Final h (after all updates) mean: {h.mean():.4f}, std: {h.std():.4f}")

    return y


def selective_scan_parallel(u, delta, A, B, C, D):
    """
    Parallel implementation (uses sequential for stability)
    In practice, this would use custom CUDA kernels
    """
    # For now, use sequential for stability
    return selective_scan_sequential(u, delta, A, B, C, D)


# =============================================================================
# TRUE MAMBA BLOCK WITH SELECTIVE STATE SPACES (STABILIZED)
# =============================================================================

class MambaBlock(nn.Module):
    """
    True Mamba block with selective state spaces (STABILIZED VERSION)

    Key fixes:
    1. Better parameter initialization
    2. Bounded activations
    3. Gradient clipping-friendly design
    """

    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = d_model * expand

        # Input projection: x -> (z, x)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # 1D Convolution for local context
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )

        # !!!!!Projection for input-dependent SSM parameters!!!!
        self.x_proj = nn.Linear(self.d_inner, self.d_inner + self.d_state * 2, bias=False)

        # Time step projection (Delta)
        self.dt_proj = nn.Linear(self.d_inner, self.d_inner, bias=True)

        # STABILITY FIX: Better initialization for A
        # Use log-space with proper scaling
        A_init = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        # Scale to prevent extreme values
        A_init = A_init / d_state
        self.A_log = nn.Parameter(torch.log(A_init))

        # D: skip connection (initialized to small values)
        self.D = nn.Parameter(torch.ones(self.d_inner) * 0.1)

        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # STABILITY: Initialize output projection with small weights
        nn.init.normal_(self.out_proj.weight, mean=0.0, std=0.02)

    def forward(self, x):
        """
        Forward pass through Mamba block (STABILIZED)
        """
        batch, seq_len, d_model = x.shape

        # Input projection
        x_and_z = self.in_proj(x)
        x, z = x_and_z.split(self.d_inner, dim=-1)

        # Apply 1D convolution
        x = x.transpose(1, 2)
        x = self.conv1d(x)[:, :, :seq_len]
        x = x.transpose(1, 2)
        # DEBUG: After 1D Conv, observe local context encoding
        # print(f"[DEBUG CONV] After 1D Conv shape: {x.shape}")

        # Activation after convolution
        x = F.silu(x)

        # !!!!Generate input-dependent SSM parameters!!!!
        x_proj = self.x_proj(x) # shorthand for self.x_proj.forward(x)

        # Split into Delta, B, C
        delta, B, C = torch.split(
            x_proj,
            [self.d_inner, self.d_state, self.d_state],
            dim=-1
        )

        # STABILITY FIX: Bounded delta with softplus and scaling
        # This prevents exploding exponentials
        delta = F.softplus(self.dt_proj(delta))
        # Scale down delta to keep it in reasonable range
        delta = delta * 0.1

        # Get A in normal space (negative for stability)
        A = -torch.exp(self.A_log)

        # SELECTIVE SCAN
        y = selective_scan_sequential(x, delta, A, B, C, self.D)
        # DEBUG: After Selective Scan, observe context output shape
        # print(f"[DEBUG SCAN] After Selective Scan output shape: {y.shape}")

        # Gated connection with bounded output
        y = y * F.silu(z)

        # Output projection
        output = self.out_proj(y)

        return output


# =============================================================================
# FULL MAMBA MODEL
# =============================================================================

class TrueMamba(nn.Module):
    """
    True Mamba language model with selective state spaces
    """

    def __init__(self, vocab_size: int, d_model: int, n_layers: int,
                 d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model

        # Token embedding
        self.embedding = nn.Embedding(vocab_size, d_model)

        # STABILITY: Better embedding initialization
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)

        # Stack of Mamba blocks
        self.layers = nn.ModuleList([
            MambaBlock(d_model, d_state, d_conv, expand)
            for _ in range(n_layers)
        ])

        # RMSNorm
        self.norm_f = RMSNorm(d_model)

        # Language modeling head
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tie weights
        self.lm_head.weight = self.embedding.weight

    def forward(self, x):
        """Forward pass"""
        # Embed tokens
        x = self.embedding(x)

        # Pass through Mamba layers with residual connections
        for layer in self.layers:
            x = x + layer(x)

        # Final normalization
        x = self.norm_f(x)

        # Output projection
        logits = self.lm_head(x)

        return logits


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""

    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        x = x / rms * self.weight
        return x


# =============================================================================
# TRAINING LOOP (WITH NAN DETECTION)
# =============================================================================

def train_model(model, dataloader, optimizer, criterion, device, num_epochs):
    """Train the Mamba model with NaN detection"""
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        num_batches = 0

        for batch_idx, (x, y) in enumerate(dataloader):
            batch_start_time = time.time() # Start timing for the batch

            # Move data to device
            x = x.to(device)
            y = y.to(device)

            # Forward pass
            logits = model(x)

            # Check for NaN in logits
            if torch.isnan(logits).any():
                print(f"WARNING: NaN detected in logits at batch {batch_idx}")
                continue

            # Reshape for loss
            logits = logits.view(-1, logits.size(-1))
            y = y.view(-1)

            # Compute loss
            loss = criterion(logits, y)

            # Check for NaN loss
            if torch.isnan(loss):
                print(f"WARNING: NaN loss at batch {batch_idx}, skipping")
                continue

            # Backward pass
            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping (important!)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Update weights
            optimizer.step()

            batch_end_time = time.time() # End timing for the batch
            batch_time = batch_end_time - batch_start_time

            # Track loss
            total_loss += loss.item()
            num_batches += 1

            # Print progress
            if (batch_idx + 1) % 50 == 0:
                avg_loss = total_loss / num_batches if num_batches > 0 else 0
                print(f"Epoch [{epoch+1}/{num_epochs}], "
                      f"Batch [{batch_idx+1}/{len(dataloader)}], "
                      f"Loss: {avg_loss:.4f}, "
                      f"Batch Time: {batch_time:.4f}s") # Print batch time

        # Epoch summary
        avg_loss = total_loss / num_batches if num_batches > 0 else 0
        print(f"Epoch [{epoch+1}/{num_epochs}] completed, Average Loss: {avg_loss:.4f}\n")


# =============================================================================
# TEXT GENERATION (WITH SAFETY CHECKS)
# =============================================================================

def generate_text(model, dataset, prompt: str, max_length: int, device, temperature: float = 1.0):
    """Generate text with safety checks"""
    model.eval()

    # Encode prompt
    input_ids = [dataset.char_to_idx[ch] for ch in prompt]
    input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

    generated = prompt

    with torch.no_grad():
        for _ in range(max_length):
            # Get logits
            logits = model(input_ids)

            # Check for NaN
            if torch.isnan(logits).any():
                print("WARNING: NaN in generation, stopping")
                break

            logits = logits[0, -1, :] / temperature

            # SAFETY: Clamp logits to prevent extreme values
            logits = torch.clamp(logits, min=-100, max=100)

            # Sample
            probs = torch.softmax(logits, dim=-1)

            # Additional safety check
            if torch.isnan(probs).any() or torch.isinf(probs).any():
                print("WARNING: Invalid probabilities, stopping")
                break

            next_token = torch.multinomial(probs, num_samples=1)

            # Append
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

            # Decode
            next_char = dataset.idx_to_char[next_token.item()]
            generated += next_char

            # Limit context window
            if input_ids.size(1) > config.seq_length:
                input_ids = input_ids[:, -config.seq_length:]

    return generated


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    print("=" * 70)
    print("TRUE MAMBA ARCHITECTURE - SHAKESPEARE TEXT GENERATION")
    print("=" * 70)
    print("\nKey Features:")
    print("  ✓ Selective state spaces (input-dependent parameters)")
    print("  ✓ Parallel/sequential scan for efficiency")
    print("  ✓ Gated connections with SiLU activation")
    print("  ✓ 1D convolution for local context")
    print("  ✓ RMSNorm for stability")
    print("  ✓ Numerical stability fixes (NaN prevention)")

    # Device info
    print(f"\nDevice: {config.device}")
    if torch.cuda.is_available():
        print(f"GPU Count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Multi-GPU: {config.use_multi_gpu}\n")

    # Load data
    text = load_shakespeare_data()
    dataset = ShakespeareDataset(text, config.seq_length, fraction=config.dataset_fraction)
    config.vocab_size = dataset.vocab_size
    print(f"Vocabulary size: {config.vocab_size} characters\n")

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0
    )

    # Create model
    print("Initializing True Mamba model...")
    model = TrueMamba(
        vocab_size=config.vocab_size,
        d_model=config.d_model,
        n_layers=config.n_layers,
        d_state=config.d_state,
        d_conv=config.d_conv,
        expand=config.expand
    )

    # Move to device
    model = model.to(config.device)

    # Multi-GPU
    if config.use_multi_gpu:
        print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    # Parameter count
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}\n")

    # Setup training
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    criterion = nn.CrossEntropyLoss()

    # Train
    print("Starting training...\n")
    train_model(model, dataloader, optimizer, criterion, config.device, config.num_epochs)

    # Generate
    print("\n" + "=" * 70)
    print("GENERATING TEXT WITH SELECTIVE SSM")
    print("=" * 70 + "\n")

    # Unwrap if multi-GPU
    generate_model = model.module if config.use_multi_gpu else model

    prompts = [
        "ROMEO:",
        "To be or not to be",
        "The king"
    ]

    for prompt in prompts:
        print(f"Prompt: '{prompt}'")
        generated = generate_text(
            generate_model,
            dataset,
            prompt,
            max_length=200,
            device=config.device,
            temperature=0.8
        )
        print(f"Generated:\n{generated}\n")
        print("-" * 70 + "\n")

if __name__ == "__main__":
    main()

TRUE MAMBA ARCHITECTURE - SHAKESPEARE TEXT GENERATION

Key Features:
  ✓ Selective state spaces (input-dependent parameters)
  ✓ Parallel/sequential scan for efficiency
  ✓ Gated connections with SiLU activation
  ✓ 1D convolution for local context
  ✓ RMSNorm for stability
  ✓ Numerical stability fixes (NaN prevention)

Device: cuda
GPU Count: 1
  GPU 0: Tesla T4
Multi-GPU: False

Loaded 1115394 characters
Using 11153 characters (1.0% of total)
Vocabulary size: 65 characters

Initializing True Mamba model...
Model parameters: 3,799,552

Starting training...

Epoch [1/5], Batch [50/333], Loss: 2.9506, Batch Time: 3.6470s
Epoch [1/5], Batch [100/333], Loss: 2.6085, Batch Time: 3.6322s
Epoch [1/5], Batch [150/333], Loss: 2.3994, Batch Time: 3.6024s
Epoch [1/5], Batch [200/333], Loss: 2.2348, Batch Time: 3.9796s
Epoch [1/5], Batch [250/333], Loss: 2.0727, Batch Time: 3.5882s
Epoch [1/5], Batch [300/333], Loss: 1.8462, Batch Time: 3.8785s
Epoch [1/5] completed, Average Loss: 1.6788

Epoch 